# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and analyze the FAIR<sup>2</sup> dataset package using the `mlcroissant` library. All dataset schema elements are referenced by their `@id` values, ensuring reproducibility and clarity when navigating the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL (see next cell).

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This section initializes the dataset object, prints a dataset summary, and prepares for further exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for FAIR^2 dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset overview
print(f"Dataset Title: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}\n")
print(f"Identifier: {dataset.metadata.identifier}\n")

## 2. Data Overview
The next step is to review which record sets are defined by the dataset schema. We'll print the main record sets and their fields by their unique `@id` values. This allows us to reference the correct layers and components in later extraction and analysis.

*Note: Use the `@id` returned in outputs to reference record sets and fields in later steps.*

In [ ]:
# List available record sets and fields with their @id
record_sets = dataset.metadata.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  description: {rs.description}")
    if rs.fields:
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}; dataType: {field.data_type})")
    print()

### Example Record Preview
Let's print a sample record from one of the record sets. Replace the `<record_set_id>` below with a real record set `@id` from the previous output. This helps map out the schema before extracting all records.

*(This cell demonstrates how to iterate and preview records by record set ID; you may need to update `<record_set_id>` to a valid value from the above cell for your use case.)*

In [ ]:
# Choose a record set @id (replace as needed, example given)
example_record_set_id = dataset.metadata.record_sets[0].id  # Use the first record set by default

print(f"Sample records for RecordSet @id: {example_record_set_id}\n")
for idx, record in enumerate(dataset.records(record_set=example_record_set_id)):
    print(record)
    if idx >= 2:
        break

## 3. Data Extraction
Load data from a specific record set into a Pandas DataFrame for further analysis. All fields and sets are always referenced by their `@id` as shown in previous steps.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records from RecordSet @id: {rs_id}")

# Preview:
chosen_rs_id = record_set_ids[0]
print(f"Columns in DataFrame for RecordSet {chosen_rs_id}:\n", dataframes[chosen_rs_id].columns.tolist())
dataframes[chosen_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
We will filter, normalize, and group data using Pandas, referring to all fields by their `@id`. Example: filter numeric values, normalize, and group by a categorical field. Modify the `numeric_field_id` and `group_field_id` as found in the DataFrame and schema above.

In [ ]:
# Reference a numeric field @id present in dataset (customize as needed)
numeric_fields = []
for field in dataset.metadata.record_sets[0].fields:
    if field.data_type in ["Float", "Integer", "Number"]:
        numeric_fields.append(field.id)

if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    numeric_field_id = dataframes[chosen_rs_id].select_dtypes(include=['number']).columns[0]  # fallback

print(f"Using numeric field for filtering: {numeric_field_id}")

# Set a threshold; adjust as appropriate for your data
threshold = 10
df = dataframes[chosen_rs_id].copy()

if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:\n", filtered_df.head())

    # Normalize the chosen numeric field
    col_mean = filtered_df[numeric_field_id].mean()
    col_std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - col_mean) / col_std
    print(f"\nNormalized {numeric_field_id} for filtered records:\n", filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field
    # Suggest a field that looks categorical
    group_fields = [f.id for f in dataset.metadata.record_sets[0].fields if f.data_type in ["Text"]]
    if group_fields:
        group_field_id = group_fields[0]
        print(f"\nGrouping filtered data by: {group_field_id}\n")
        grouped = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(grouped.head())
    else:
        print("No categorical field found for grouping.")
else:
    print(f"Column {numeric_field_id} not found in DataFrame.")

## 5. Visualization
Visualize the distribution of the selected numeric field using matplotlib. You can adjust the fields based on what's available in the DataFrame extracted above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
else:
    print(f"Field {numeric_field_id} not found for visualization.")

## 6. Conclusion

- Loaded a FAIR<sup>2</sup>-conformant protein mass spectrometry dataset via a Croissant schema.
- Reviewed dataset structure, record sets, and fields (with `@id` references for precision and reproducibility).
- Demonstrated extracting, filtering, and normalizing a numeric field and visualized its distribution.
- This workflow can be adapted for other Croissant-structured datasets for scalable, FAIR data science.